# HOU53-bot — Preprocessing

In this notebook we will preprocess the Ames Housing dataset based on the findings of the EDA. Steps covered:
1. Drop useless columns
2. Handle missing values
3. Remove outliers
4. Log-transform skewed features
5. Encode categorical variables

## Library Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew, mstats
import os

## Data Load

In [2]:
INPUT_DATA_FILE = "data/raw/house_prices.csv"
OUTPUT_DATA_FILE = "data/processed/house_prices_preprocessed.csv"

## Load Data

In [3]:
df = pd.read_csv(INPUT_DATA_FILE, na_values=['?', 'NA'])

display(df)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


## Null Values

In [4]:
cat_none_cols = [
    'Fence',         # no fence
    'FireplaceQu',   # no fireplace
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',  # no garage
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',  # no basement
    'MasVnrType',    # no masonry veneer
    'Alley',         # no alley access
    'PoolQC',        # no pool
    'MiscFeature',   # no misc feature
]
for col in cat_none_cols:
    df[col] = df[col].fillna('None')

In [5]:
num_zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
    'MasVnrArea',
]
for col in num_zero_cols:
    df[col] = df[col].fillna(0)

In [6]:
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'] \
                      .transform(lambda x: x.fillna(x.median()))

df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [7]:
nulls = df.isnull().sum().sort_values(ascending=False) # Check for null values and sort them in descending order

pd.DataFrame({'Nulls': nulls, '% of nulls': (nulls / len(df) * 100).round(2)})

,Nulls,% of nulls
Id,0,0.0
MSSubClass,0,0.0
MSZoning,0,0.0
LotFrontage,0,0.0
LotArea,0,0.0
...,...,...
MoSold,0,0.0
YrSold,0,0.0
SaleType,0,0.0
SaleCondition,0,0.0


## Column Treatment

In [8]:
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}

ordinal_quality_cols = [
    'ExterQual', 'ExterCond',
    'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual',
    'FireplaceQu',
    'GarageQual', 'GarageCond',
]
for col in ordinal_quality_cols:
    df[col] = df[col].map(quality_map)

In [9]:
# Convert to string first to ensure get_dummies treats them as categorical
df['MSSubClass'] = df['MSSubClass'].astype(str)
df = pd.get_dummies(df, columns=['MSSubClass'], drop_first=True)

In [10]:
df['BsmtExposure'] = df['BsmtExposure'].map({'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0})

bsmt_fin_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
df['BsmtFinType1'] = df['BsmtFinType1'].map(bsmt_fin_map)
df['BsmtFinType2'] = df['BsmtFinType2'].map(bsmt_fin_map)

df['GarageFinish'] = df['GarageFinish'].map({'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0})

df['Functional'] = df['Functional'].map(
    {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
)

df['LandSlope'] = df['LandSlope'].map({'Gtl': 2, 'Mod': 1, 'Sev': 0})

df['PavedDrive'] = df['PavedDrive'].map({'Y': 2, 'P': 1, 'N': 0})

df['CentralAir'] = df['CentralAir'].map({'Y': 1, 'N': 0})

In [11]:
df.drop(columns=['Utilities', 'Id'], inplace=True)
print(f'Shape: {df.shape}')

Shape: (1460, 92)


In [12]:
nominal_cols = [
    'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour',
    'LotConfig', 'Neighborhood', 'Condition1', 'Condition2',
    'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl',
    'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation',
    'Heating', 'Electrical', 'GarageType', 'MiscFeature',
    'SaleType', 'SaleCondition', 'Fence', 'PoolQC',
]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

In [13]:
print(f'Shape after encoding: {df.shape}')

display(df)

Shape after encoding: (1460, 222)


,LotFrontage,LotArea,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,...,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,Fence_GdWo,Fence_MnPrv,Fence_MnWw,Fence_None,PoolQC_Fa,PoolQC_Gd,PoolQC_None
0,65.0,8450,2,7,5,2003,2003,196.0,4,3,...,False,True,False,False,False,False,True,False,False,True
1,80.0,9600,2,6,8,1976,1976,0.0,3,3,...,False,True,False,False,False,False,True,False,False,True
2,68.0,11250,2,7,5,2001,2002,162.0,4,3,...,False,True,False,False,False,False,True,False,False,True
3,60.0,9550,2,7,5,1915,1970,0.0,3,3,...,False,False,False,False,False,False,True,False,False,True
4,84.0,14260,2,8,5,2000,2000,350.0,4,3,...,False,True,False,False,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,62.0,7917,2,6,5,1999,2000,0.0,3,3,...,False,True,False,False,False,False,True,False,False,True
1456,85.0,13175,2,6,6,1978,1988,119.0,3,3,...,False,True,False,False,True,False,False,False,False,True
1457,66.0,9042,2,7,9,1941,2006,0.0,5,4,...,False,True,False,False,False,False,False,False,False,True
1458,68.0,9717,2,5,6,1950,1996,0.0,3,3,...,False,True,False,False,False,False,True,False,False,True


## Outliers

In [14]:
before = len(df)
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 200000))]
print(f'Removed {before - len(df)} rows. New shape: {df.shape}')

Removed 2 rows. New shape: (1458, 222)


In [15]:
cols_to_winsorize = ['LotArea', 'TotalBsmtSF', 'GarageArea', '1stFlrSF']
for col in cols_to_winsorize:
    df[col] = mstats.winsorize(df[col], limits=[0.01, 0.01])

## Feature Engineering

In [16]:
df['TotalQual'] = df['OverallQual'] * df['OverallCond']
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
df['WasRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
df['TotalBaths'] = (df['FullBath'] + df['BsmtFullBath'] + 0.5 * (df['HalfBath'] + df['BsmtHalfBath']))
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
df['HasPool'] = (df['PoolArea'] > 0).astype(int)
df['HasPorch'] = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                      df['ScreenPorch'] + df['WoodDeckSF'] > 0).astype(int)
df['QualPerSF'] = df['OverallQual'] / (df['GrLivArea'] + 1)
df['Qual_x_LiveArea'] = df['OverallQual'] * df['GrLivArea']
df['Qual_x_TotalSF'] = df['OverallQual'] * df['TotalSF']
df['Qual_x_GarageArea']= df['OverallQual'] * df['GarageArea']
df['BsmtFinRatio'] = df['BsmtFinSF1'] / (df['TotalBsmtSF'] + 1)
df['GarageAge']= df['YrSold'] - df['GarageYrBlt']
df['GarageNewerThanHouse'] = (df['GarageYrBlt'] > df['YearBuilt']).astype(int)
df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                      df['ScreenPorch'] + df['WoodDeckSF'] + df['3SsnPorch'])

## Target

In [17]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop('SalePrice')
skewness = df[numeric_cols].apply(skew).sort_values(ascending=False)

print(skewness[skewness > 0.75])  # Threshold

MiscVal                 24.434913
PoolArea                15.932532
HasPool                 15.492067
3SsnPorch               10.286510
LowQualFinSF             8.995688
KitchenAbvGr             4.480268
BsmtFinSF2               4.247550
ScreenPorch              4.114690
BsmtHalfBath             4.095895
GarageAge                3.862350
BsmtFinType2             3.290761
EnclosedPorch            3.083987
MasVnrArea               2.693554
OpenPorchSF              2.337421
LotArea                  2.232488
GarageNewerThanHouse     1.557997
LotFrontage              1.546174
WoodDeckSF               1.544214
Qual_x_LiveArea          1.492289
ExterCond                1.394028
Qual_x_TotalSF           1.215277
BsmtExposure             1.106364
TotalPorchSF             1.103269
GrLivArea                1.009951
BsmtUnfSF                0.919955
Qual_x_GarageArea        0.837206
ExterQual                0.819203
2ndFlrSF                 0.812121
BsmtFinSF1               0.764002
dtype: float64

In [18]:
cols_to_log = [
    'MasVnrArea',
    'OpenPorchSF',
    'LotArea',
    'LotFrontage',
    'WoodDeckSF',
    'GrLivArea',
    'BsmtUnfSF',
    '2ndFlrSF',
    'BsmtFinSF1',
    'EnclosedPorch',
]

df[cols_to_log] = np.log1p(df[cols_to_log])

# Target
df['SalePrice'] = np.log1p(df['SalePrice'])

In [19]:
display(df)

,LotFrontage,LotArea,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,...,HasPool,HasPorch,QualPerSF,Qual_x_LiveArea,Qual_x_TotalSF,Qual_x_GarageArea,BsmtFinRatio,GarageAge,GarageNewerThanHouse,TotalPorchSF
0,4.189655,9.042040,2,7,5,2003,2003,5.283204,4,3,...,0,1,0.004091,11970,17962,3836,0.823804,5.0,0,61
1,4.394449,9.169623,2,6,8,1976,1976,0.000000,3,3,...,0,1,0.004751,7572,15144,2760,0.774347,31.0,0,298
2,4.234107,9.328212,2,7,5,2001,2002,5.093750,4,3,...,0,1,0.003917,12502,18942,4256,0.527687,7.0,0,42
3,4.110874,9.164401,2,7,5,1915,1970,0.000000,3,3,...,0,1,0.004075,12019,17311,4494,0.285337,8.0,1,307
4,4.442651,9.565284,2,8,5,2000,2000,5.860786,4,3,...,0,1,0.003638,17584,26744,6688,0.571553,8.0,0,276
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,4.143135,8.976894,2,6,5,1999,2000,0.000000,3,3,...,0,1,0.003641,9882,15600,2760,0.000000,8.0,0,40
1456,4.454347,9.486152,2,6,6,1978,1988,4.787492,3,3,...,0,1,0.002893,12438,21690,3000,0.511990,32.0,0,349
1457,4.204693,9.109746,2,7,9,1941,2006,0.000000,5,4,...,0,1,0.002990,16380,24444,1764,0.238508,69.0,0,60
1458,4.234107,9.181735,2,5,6,1950,1996,0.000000,3,3,...,0,1,0.004634,5390,10780,1200,0.045412,60.0,0,478


## Save Preprocessed Data

In [20]:
os.makedirs('data/processed', exist_ok=True)

df.to_csv(OUTPUT_DATA_FILE, index=False)

print(f'Saved: {OUTPUT_DATA_FILE}')
print(f'Shape: {df.shape}')
df.head()

Saved: data/processed/house_prices_preprocessed.csv
Shape: (1458, 241)


,LotFrontage,LotArea,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,...,HasPool,HasPorch,QualPerSF,Qual_x_LiveArea,Qual_x_TotalSF,Qual_x_GarageArea,BsmtFinRatio,GarageAge,GarageNewerThanHouse,TotalPorchSF
0,4.189655,9.042040,2,7,5,2003,2003,5.283204,4,3,...,0,1,0.004091,11970,17962,3836,0.823804,5.0,0,61
1,4.394449,9.169623,2,6,8,1976,1976,0.000000,3,3,...,0,1,0.004751,7572,15144,2760,0.774347,31.0,0,298
2,4.234107,9.328212,2,7,5,2001,2002,5.093750,4,3,...,0,1,0.003917,12502,18942,4256,0.527687,7.0,0,42
3,4.110874,9.164401,2,7,5,1915,1970,0.000000,3,3,...,0,1,0.004075,12019,17311,4494,0.285337,8.0,1,307
4,4.442651,9.565284,2,8,5,2000,2000,5.860786,4,3,...,0,1,0.003638,17584,26744,6688,0.571553,8.0,0,276
